In [1]:
import pandas as pd
import os

os.chdir("d:/Projects/volatility-radar")

df = pd.read_csv("data/processed/full_merged_clean.csv")

print(df.dtypes)
df.head()

date                str
time                str
currency            str
event               str
impact              str
pair                str
open            float64
high            float64
low             float64
close           float64
actual          float64
previous        float64
surprise        float64
impact_score      int64
dtype: object


,date,time,currency,event,impact,pair,open,high,low,close,actual,previous,surprise,impact_score
0,2021-01-04,1:45pm,EUR,Spanish Manufacturing PMI,Low,EURUSD,1.22386,1.23098,1.2229,1.22461,51.0,49.8,1.2,1
1,2021-01-04,2:15pm,EUR,Italian Manufacturing PMI,Low,EURUSD,1.22386,1.23098,1.2229,1.22461,52.8,51.5,1.3,1
2,2021-01-04,2:20pm,EUR,French Final Manufacturing PMI,Low,EURUSD,1.22386,1.23098,1.2229,1.22461,51.1,51.1,0.0,1
3,2021-01-04,2:25pm,EUR,German Final Manufacturing PMI,Low,EURUSD,1.22386,1.23098,1.2229,1.22461,58.3,58.6,-0.3,1
4,2021-01-04,2:30pm,EUR,Final Manufacturing PMI,Low,EURUSD,1.22386,1.23098,1.2229,1.22461,55.2,55.5,-0.3,1


In [2]:
daily = df[['date', 'pair', 'open', 'high', 'low', 'close']].drop_duplicates(subset = ['date', 'pair'])

In [3]:
daily = daily.sort_values(['pair', 'date']).reset_index(drop = True)

In [4]:
daily.head()

,date,pair,open,high,low,close
0,2021-01-04,EURUSD,1.22386,1.23098,1.22290,1.22461
1,2021-01-05,EURUSD,1.22469,1.23057,1.22428,1.22957
2,2021-01-06,EURUSD,1.22960,1.23495,1.22640,1.23242
3,2021-01-07,EURUSD,1.23250,1.23445,1.22430,1.22688
4,2021-01-08,EURUSD,1.22705,1.22848,1.21930,1.21970


In [5]:
daily[['next_open', 'next_high', 'next_low', 'next_close']] = daily.groupby('pair')[['open', 'high', 'low', 'close']].shift(-1)

In [6]:
daily.head()

,date,pair,open,high,low,close,next_open,next_high,next_low,next_close
0,2021-01-04,EURUSD,1.22386,1.23098,1.22290,1.22461,1.22469,1.23057,1.22428,1.22957
1,2021-01-05,EURUSD,1.22469,1.23057,1.22428,1.22957,1.22960,1.23495,1.22640,1.23242
2,2021-01-06,EURUSD,1.22960,1.23495,1.22640,1.23242,1.23250,1.23445,1.22430,1.22688
3,2021-01-07,EURUSD,1.23250,1.23445,1.22430,1.22688,1.22705,1.22848,1.21930,1.21970
4,2021-01-08,EURUSD,1.22705,1.22848,1.21930,1.21970,1.22160,1.22284,1.21310,1.21495


In [7]:
daily['return'] = (daily['close'] - daily['open']) * 100/daily['open']

In [8]:
daily[['date', 'pair', 'return']].head()

,date,pair,return
0,2021-01-04,EURUSD,0.061282
1,2021-01-05,EURUSD,0.398468
2,2021-01-06,EURUSD,0.229343
3,2021-01-07,EURUSD,-0.455984
4,2021-01-08,EURUSD,-0.598998


In [9]:
daily['rolling_std'] = daily.groupby('pair')['return'].transform(
    lambda x : x.rolling(20, min_periods = 10).std()
)
daily.head(15)

,date,pair,open,high,low,close,next_open,next_high,next_low,next_close,return,rolling_std
0,2021-01-04,EURUSD,1.22386,1.23098,1.22290,1.22461,1.22469,1.23057,1.22428,1.22957,0.061282,NaN
1,2021-01-05,EURUSD,1.22469,1.23057,1.22428,1.22957,1.22960,1.23495,1.22640,1.23242,0.398468,NaN
2,2021-01-06,EURUSD,1.22960,1.23495,1.22640,1.23242,1.23250,1.23445,1.22430,1.22688,0.229343,NaN
3,2021-01-07,EURUSD,1.23250,1.23445,1.22430,1.22688,1.22705,1.22848,1.21930,1.21970,-0.455984,NaN
4,2021-01-08,EURUSD,1.22705,1.22848,1.21930,1.21970,1.22160,1.22284,1.21310,1.21495,-0.598998,NaN
5,2021-01-11,EURUSD,1.22160,1.22284,1.21310,1.21495,1.21505,1.22097,1.21350,1.22052,-0.544368,NaN
6,2021-01-12,EURUSD,1.21505,1.22097,1.21350,1.22052,1.22057,1.22229,1.21380,1.21565,0.450187,NaN
7,2021-01-13,EURUSD,1.22057,1.22229,1.21380,1.21565,1.21566,1.21788,1.21100,1.21515,-0.403090,NaN
8,2021-01-14,EURUSD,1.21566,1.21788,1.21100,1.21515,1.21540,1.21625,1.20720,1.20749,-0.041953,NaN
9,2021-01-15,EURUSD,1.21540,1.21625,1.20720,1.20749,1.20744,1.21450,1.20720,1.21280,-0.650815,0.425123


In [10]:
daily.isnull().sum()

date            0
pair            0
open            0
high            0
low             0
close           0
next_open       3
next_high       3
next_low        3
next_close      3
return          0
rolling_std    27
dtype: int64

In [11]:
import numpy as np
def assign_label(row):
    if pd.isna(row['return']) or pd.isna(row['rolling_std']):
        return np.nan
    
    move = row['return']
    
    t1 = 0.35 * row['rolling_std']  # medium threshold
    t2 = 0.75 * row['rolling_std']  # strong threshold
    t3 = 1.25 * row['rolling_std']  # very strong threshold
    
    if move > t3:
        return 6   # very strong up
    elif move > t2:
        return 5   # strong up
    elif move > t1:
        return 4   # medium up
    elif move < -t3:
        return 0   # very strong down
    elif move < -t2:
        return 1   # strong down
    elif move < -t1:
        return 2   # medium down
    else:
        return 3   # no move

daily['label'] = daily.apply(assign_label, axis=1)

In [12]:
print(daily['label'].value_counts().sort_index())

label
0.0     388
1.0     407
2.0     522
3.0    1102
4.0     539
5.0     460
6.0     412
Name: count, dtype: int64


In [13]:
daily.dtypes

date               str
pair               str
open           float64
high           float64
low            float64
close          float64
next_open      float64
next_high      float64
next_low       float64
next_close     float64
return         float64
rolling_std    float64
label          float64
dtype: object